# Load Python Libraries and Dataset

In [0]:
!pip install openai
dbutils.library.restartPython()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

import os
import base64
import json
import re
from openai import OpenAI
import mlflow
mlflow.openai.autolog()

In [0]:
PATH = '/Volumes/datascience/master_metrics/stragegy_dataset/tier1/final_tier1_percentiles.parquet'
df = spark.read.parquet(PATH)
df = df.toPandas()
df.head()

,zipcode,county_name,county_fips,state_fips,state_name,msa,msa_name,age_65_plus_pct,population_growth_rate_2yr,age_45_64_pct,birth_rate_per_1000,total_population,age_65_plus_pct_pctile,population_growth_rate_2yr_pctile,age_45_64_pct_pctile,birth_rate_per_1000_pctile,total_population_pctile
0,30002,DeKalb,13089,13,Georgia,Yes,"Atlanta-Sandy Springs-Alpharetta, GA",1.7,-10.78,37.2,60.1,5225,30.0,13.0,62.0,70.0,47
1,30004,Fulton,13121,13,Georgia,Yes,"Atlanta-Sandy Springs-Alpharetta, GA",2.4,3.47,36.5,43.9,68707,55.0,67.0,29.0,51.0,98
2,30005,Fulton,13121,13,Georgia,Yes,"Atlanta-Sandy Springs-Alpharetta, GA",1.4,3.91,37.8,42.2,41836,21.0,69.0,81.0,49.0,89
3,30008,Cobb,13067,13,Georgia,Yes,"Atlanta-Sandy Springs-Alpharetta, GA",1.6,3.54,36.8,36.9,35547,26.0,67.0,42.0,43.0,85
4,30009,Fulton,13121,13,Georgia,Yes,"Atlanta-Sandy Springs-Alpharetta, GA",2.3,-0.87,36.7,70.4,21095,51.0,41.0,37.0,79.0,70


# Compute Attractiveness Score

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── variables included in Attractiveness scoring ─────────────────────────────
ATTRACT_VARS = [
    "total_population",
    "population_growth_rate_2yr",
    "age_65_plus_pct",
    "age_45_64_pct",
    "birth_rate_per_1000",
]
N = len(ATTRACT_VARS)  # = 5

In [0]:
# ============================================================
# OPTION 1 — Equal-Weight Linear Additive (Z-Score Normalized)
# Formula: A = (1/N) * Σ Z_i   rescaled to 0–100
# Z_i = (x_i - mean_i) / std_i  across all ZIPs
# ============================================================

df_scored = df

# Step 1: Z-score each variable
z_scores = pd.DataFrame()
for var in ATTRACT_VARS:
    z_scores[var] = (df[var] - df[var].mean()) / df[var].std()

# Step 2: equal-weight average
z_avg = z_scores.mean(axis=1)

# Step 3: rescale to 0–100
df["attractiveness_score_opt1"] = (
    (z_avg - z_avg.min()) / (z_avg.max() - z_avg.min()) * 100
).round(2)


In [0]:
# =============================================================
# OPTION 2 — Expert weights — must sum to 1.0
# Formula: A = Σ (w_i * percentile_i)
# Percentiles are already on a 1-99 scale, so no normalization needed.
# Final score is rescaled to 0-100 by dividing by 99.
# =============================================================

WEIGHTS = {
    "age_65_plus_pct_pctile":            0.20,
    "population_growth_rate_2yr_pctile": 0.15,
    "age_45_64_pct_pctile":              0.10,
    "birth_rate_per_1000_pctile":        0.05,
    "total_population_pctile":           0.50,
}

# Sanity check
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, \
    f"Weights must sum to 1.0 — current sum: {sum(WEIGHTS.values()):.4f}"

# Step 1: Apply expert weights directly to the 5 percentile columns
weighted_sum = sum(
    df[col] * w
    for col, w in WEIGHTS.items()
)

# Step 2: Rescale to 0-100 by dividing by 99
df["attractiveness_score_opt2"] = weighted_sum.round(2)   # stays 0-99

print(df["attractiveness_score_opt2"].describe())

count    2298.000000
mean       54.463011
std        14.712964
min         3.350000
25%        45.200000
50%        56.400000
75%        65.500000
max        88.100000
Name: attractiveness_score_opt2, dtype: float64


In [0]:
# =============================================================
# OPTION 4 — Equal-weight Percentile Rank (Non-Parametric)
# Formula: A = (1/N) * Σ P_i   where P_i = percentile rank 0-100
# Uses the same 5 Neil-selected columns — equal weight (0.20 each)
# =============================================================

ATTRACT_VARS = [
    "age_65_plus_pct_pctile",
    "population_growth_rate_2yr_pctile",
    "age_45_64_pct_pctile",
    "birth_rate_per_1000_pctile",
    "total_population_pctile",
]

# Step 1: These are already percentile columns (0-99 scale)
#         No re-ranking needed — use directly
pct_ranks = df[ATTRACT_VARS].copy()

# Step 2: Equal-weight average — rescale to 0-100
df["attractiveness_score_opt4"] = pct_ranks.mean(axis=1).round(2)  # stays 0-99

print(df["attractiveness_score_opt4"].describe())

count    2377.000000
mean       51.547270
std        13.319063
min         1.000000
25%        45.400000
50%        54.000000
75%        60.600000
max        86.200000
Name: attractiveness_score_opt4, dtype: float64


In [0]:
# ── preview top 20 ZIPs by Option 1 score ────────────────────────────────────
display(
    df[["zipcode"] + ATTRACT_VARS +
       ["attractiveness_score_opt1","attractiveness_score_opt2", "attractiveness_score_opt4"]]
    .sort_values("attractiveness_score_opt2", ascending=False)
    .head(20)
)

zipcode,age_65_plus_pct_pctile,population_growth_rate_2yr_pctile,age_45_64_pct_pctile,birth_rate_per_1000_pctile,total_population_pctile,attractiveness_score_opt1,attractiveness_score_opt2,attractiveness_score_opt4
32162,80.0,73.0,90.0,93.0,95,64.98,88.1,86.2
33027,89.0,63.0,66.0,44.0,98,62.58,85.05,72.0
34748,77.0,84.0,70.0,68.0,93,61.18,84.9,78.4
34135,82.0,74.0,73.0,60.0,93,61.03,84.3,76.4
32164,71.0,87.0,66.0,53.0,95,61.49,84.0,74.4
34293,92.0,80.0,42.0,41.0,93,61.03,83.15,69.6
34655,67.0,86.0,62.0,79.0,93,61.36,82.95,77.4
34120,82.0,86.0,52.0,61.0,90,60.2,82.55,74.2
33411,87.0,69.0,29.0,47.0,99,63.11,82.5,66.2
34711,71.0,74.0,62.0,37.0,98,62.1,82.35,68.4


# Set up OpenAI API Call Configuration

In [0]:

 # Get workspace URL automatically
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
 
# Get internal notebook token
notebook_token = dbutils.notebook.entry_point.getDbutils()\
    .notebook().getContext().apiToken().get()
 
client = OpenAI(
    api_key=notebook_token,
    base_url=f"https://adb-1956198073399317.17.azuredatabricks.net/serving-endpoints"
)

# Set Up the Market Analytics AI Agent

In [0]:
def execute_llm_generated_code(df: pd.DataFrame, user_question: str) -> str:
    """
    Pattern: LLM sees column schema only → writes Pandas code → 
    code runs locally against full dataframe → result sent back to LLM → 
    LLM writes final answer.
    """

    # ── Step 1: Give LLM the schema + sample, NOT the full data ──────────────
    schema_context = f"""
                        You have access to a pandas DataFrame called `df` with the following structure:

                        COLUMNS:
                        {df.dtypes.to_string()}

                        SHAPE: {df.shape[0]} rows × {df.shape[1]} columns

                        SAMPLE (first 3 rows):
                        {df.head(3).to_csv(index=False)}

                        STATISTICS:
                        {df.describe().to_string()}
                        """

    # ── Step 2: Ask LLM to write Pandas code to answer the question ──────────
    code_generation_prompt = f"""
    {schema_context}

    User question: "{user_question}"

    Write Python/Pandas code to answer this question using the DataFrame `df`.
    Return ONLY executable Python code, no explanation, no markdown fences.
    Store your final answer in a variable called `result`.
    If the exact column doesn't exist, use the closest available columns and 
    add a comment explaining your assumption.
    """

    code_response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a Python/Pandas expert. Return only raw executable code."},
            {"role": "user",   "content": code_generation_prompt}
        ],
        max_tokens=512,
        temperature=0.1
    )

    generated_code = code_response.choices[0].message.content
    #print("── LLM Generated Code ──────────────────────────────")
    #print(generated_code)
    #print("────────────────────────────────────────────────────")

    # ── Step 3: Execute the generated code against the REAL dataframe ─────────
    local_vars = {"df": df}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", "No result variable found.")
    except Exception as e:
        result = f"Code execution error: {e}"

    #print(f"\n── Code Execution Result ───────────────────────────")
    #print(result)
    #print("────────────────────────────────────────────────────")

    # ── Step 4: Send result back to LLM for a natural language answer ─────────
    final_response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a strategic healthcare market analyst for Orlando Health, writing for senior executives. Every response MUST follow this exact structure:\n1. HEADLINE (one bold sentence): the direct answer or bottom-line conclusion.\n2. SUPPORTING BULLETS: 3-5 concise bullet points with the key facts, numbers, and context.\n3. STRATEGIC IMPLICATION (one bold sentence): what this means for Orlando Health.\n4. SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next.\nUse dashes only in hyphenated words (e.g. part-time); use colons and semicolons to separate clauses, never dashes."},
            {"role": "user",   "content": (
                f"Original question: \"{user_question}\"\\n\\n"
                f"The Pandas code returned this result:\\n{result}\\n\\n"
                f"Answer using this exact structure:\\n"
                f"HEADLINE: One bold sentence with the direct answer.\\n"
                f"SUPPORTING BULLETS: 3-5 bullets with key facts and numbers.\\n"
                f"STRATEGIC IMPLICATION: One bold sentence on what this means for Orlando Health.\\n"
                f"SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next."
            )}
        ],
        max_tokens=400,
        temperature=0.3
    )

    return final_response.choices[0].message.content

# User Demo - Interactive Talk with the Agent

## Demo 1 - On the Surface Questions

In [0]:
questions = [
    "Which zip code has the highest population growth rate??"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    print('='*60)
    answer = execute_llm_generated_code(df, q)
    print(f"\nFINAL ANSWER: {answer}")


QUESTION: Which zip code has the highest population growth rate??

FINAL ANSWER: **HEADLINE: Zip code 30289 has the highest population growth rate in the dataset, with a remarkable 843.48% two-year growth rate.**

SUPPORTING BULLETS:
- Zip code 30289 recorded a two-year population growth rate of 843.48%, far exceeding typical organic growth patterns in most markets
- Growth of this magnitude often signals a newly developed community, a large residential construction project, or a data reclassification rather than purely organic migration
- This zip code ranks 166th in the dataset (index position 165), suggesting it may represent a smaller or emerging geography rather than an established urban core
- Extreme growth rates warrant validation: the underlying population base may be very small, making percentage swings disproportionately large
- If the growth is genuine, this area represents a rapidly expanding patient population with potentially unmet healthcare needs

**STRATEGIC IMPLICAT

[Trace(trace_id=tr-d9bcabaf16afdd3cded03895e8f6bf38), Trace(trace_id=tr-82854c82ccc75322d2ddc8b8fb0224c7)]

In [0]:
questions = [
    "What is the zip code with highest attractiveness score under option 2?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    print('='*60)
    answer = execute_llm_generated_code(df, q)
    print(f"\nFINAL ANSWER: {answer}")


QUESTION: What is the zip code with highest attractiveness score under option 2?

FINAL ANSWER: **HEADLINE: Under Option 2, ZIP code 32162 holds the highest attractiveness score, making it the top priority target market for expansion consideration.**

SUPPORTING BULLETS:
- ZIP code 32162 corresponds to the Villages area in Sumter/Marion County, a rapidly growing retirement community with one of the fastest expanding senior populations in the nation
- Senior demographics in this ZIP code align strongly with high utilization services such as orthopedics, cardiovascular care, and oncology: all core Orlando Health service lines
- Option 2's attractiveness scoring methodology likely weights factors such as population growth, payer mix, competitor density, and access gaps; 32162 scores highest across this composite index
- The Villages market has historically been underserved by major health systems, creating a meaningful white space opportunity for a well resourced competitor
- Geographic 

[Trace(trace_id=tr-773a5f1672db5593b7e0657e50ec3127), Trace(trace_id=tr-0e59f0fed37cca2d73ed8aee58aacee1)]

## Demo 2 - Aggregated Aanalytics

In [0]:
questions = [
    "Which MSA has the highest on average attractiness score under option 2??"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    print('='*60)
    answer = execute_llm_generated_code(df, q)
    print(f"\nFINAL ANSWER: {answer}")


QUESTION: Which MSA has the highest on average attractiness score under option 2??

FINAL ANSWER: **HEADLINE: Under Option 2, Sebastian-Vero Beach, FL ranks as the highest scoring MSA on average attractiveness, making it the most compelling expansion candidate by this methodology.**

SUPPORTING BULLETS:
- Sebastian-Vero Beach, FL achieved the top average attractiveness score among all evaluated MSAs under Option 2's scoring framework
- This MSA, located in Indian River County along Florida's Treasure Coast, sits within reasonable proximity to Orlando Health's existing service footprint
- Option 2's scoring methodology likely weights a specific combination of market factors (such as population growth, competition density, or payer mix) that favor this market's profile
- Sebastian-Vero Beach represents a mid-sized, growing coastal market with demographic trends (aging population, in-migration) that typically align with healthcare demand growth
- The result warrants validation against Op

[Trace(trace_id=tr-b7063ae3a22fe73ec3ecbc3c2dccc7b3), Trace(trace_id=tr-2706badf62dd3ca068d9e21521623990)]

In [0]:
questions = [
    "Which MSA within the stage of Alabama has the highest on average attractiness score under option 2??"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    print('='*60)
    answer = execute_llm_generated_code(df, q)
    print(f"\nFINAL ANSWER: {answer}")


QUESTION: Which MSA within the stage of Alabama has the highest on average attractiness score under option 2??
MSA with highest average attractiveness score (opt2) in Alabama: Huntsville, AL
Average score: 56.13

All Alabama MSAs ranked:
msa_name
Huntsville, AL                56.132258
Daphne-Fairhope-Foley, AL     54.747500
Florence-Muscle Shoals, AL    53.132143
Anniston-Oxford, AL           52.516667
Gadsden, AL                   52.510000
Talladega-Sylacauga, AL       51.443750
Decatur, AL                   51.153333
Birmingham-Hoover, AL         50.359821
Tuscaloosa, AL                49.476087
Auburn-Opelika, AL            49.259091
Montgomery, AL                47.538462
Dothan, AL                    47.360870
Mobile, AL                    46.453191
Columbus, GA-AL               43.964286
Name: attractiveness_score_opt2, dtype: float64

FINAL ANSWER: **HEADLINE: Huntsville, AL ranks as the highest attractiveness MSA in Alabama under Option 2, making it the most strategically co

[Trace(trace_id=tr-0f1ffa069a630a2ad9f75a8c901b70bd), Trace(trace_id=tr-741f3052b0c3dbab9f7dd3976a0d0161)]

In [0]:
questions = [
    "What is the avreage attractiveness score under option 2 for birmingham-hoover MSA??"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    print('='*60)
    answer = execute_llm_generated_code(df, q)
    print(f"\nFINAL ANSWER: {answer}")


QUESTION: What is the avreage attractiveness score under option 2 for birmingham-hoover MSA??

FINAL ANSWER: **HEADLINE: Under Option 2, the Birmingham-Hoover MSA posts an average attractiveness score of 50.36 out of 100, indicating a moderate market opportunity.**

SUPPORTING BULLETS:
- The average attractiveness score of 50.36 places Birmingham-Hoover in the middle tier of evaluated markets, neither a standout opportunity nor a clear disqualifier.
- Option 2 likely reflects a specific weighting methodology or market entry scenario; this score should be benchmarked against other MSAs evaluated under the same option to determine relative ranking.
- Birmingham-Hoover is Alabama's largest metro area, with a population exceeding 1.1 million; a mid-range score suggests mixed performance across underlying dimensions such as population growth, payer mix, or competitive density.
- A score near 50 warrants deeper decomposition: identifying which sub-criteria are dragging the score down versus

[Trace(trace_id=tr-4075d209b2522135458f866f9d7924a5), Trace(trace_id=tr-001f34cae69c48befb70fcfc53e4dfd6)]

# Advanced Strategy AI Agent

## Agent Configuration

In [0]:
# =============================================================
# SCORE_DEFINITIONS — Neil's 5 selected percentile columns
# Used by: explain engine, what-if engine, comparison engine
# All 3 options use the same 5 columns, only weights differ
# =============================================================

SCORE_DEFINITIONS = {
    "attractiveness_score_opt1": {
        "description": "Option 1: Neil's 5-factor score (equal weights)",
        "components": {
            "total_population_pctile":           {"weight": 0.20, "direction": "higher_is_better", "label": "Total Population"},
            "age_65_plus_pct_pctile":            {"weight": 0.20, "direction": "higher_is_better", "label": "Age 65+ Share (%)"},
            "population_growth_rate_2yr_pctile": {"weight": 0.20, "direction": "higher_is_better", "label": "Population Growth Rate (2yr)"},
            "age_45_64_pct_pctile":              {"weight": 0.20, "direction": "higher_is_better", "label": "Age 45-64 Share (%)"},
            "birth_rate_per_1000_pctile":        {"weight": 0.20, "direction": "higher_is_better", "label": "Birth Rate (per 1,000)"},
        }
    },
    "attractiveness_score_opt2": {
        "description": "Option 2: Neil's 5-factor score (expert weights)",
        "components": {
            "total_population_pctile":           {"weight": 0.50, "direction": "higher_is_better", "label": "Total Population"},
            "age_65_plus_pct_pctile":            {"weight": 0.20, "direction": "higher_is_better", "label": "Age 65+ Share (%)"},
            "population_growth_rate_2yr_pctile": {"weight": 0.15, "direction": "higher_is_better", "label": "Population Growth Rate (2yr)"},
            "age_45_64_pct_pctile":              {"weight": 0.10, "direction": "higher_is_better", "label": "Age 45-64 Share (%)"},
            "birth_rate_per_1000_pctile":        {"weight": 0.05, "direction": "higher_is_better", "label": "Birth Rate (per 1,000)"},
        }
    },
    "attractiveness_score_opt4": {
        "description": "Option 4: Neil's 5-factor score (equal-weight percentile rank)",
        "components": {
            "total_population_pctile":           {"weight": 0.20, "direction": "higher_is_better", "label": "Total Population"},
            "age_65_plus_pct_pctile":            {"weight": 0.20, "direction": "higher_is_better", "label": "Age 65+ Share (%)"},
            "population_growth_rate_2yr_pctile": {"weight": 0.20, "direction": "higher_is_better", "label": "Population Growth Rate (2yr)"},
            "age_45_64_pct_pctile":              {"weight": 0.20, "direction": "higher_is_better", "label": "Age 45-64 Share (%)"},
            "birth_rate_per_1000_pctile":        {"weight": 0.20, "direction": "higher_is_better", "label": "Birth Rate (per 1,000)"},
        }
    },
}

# Mapping: raw column → precomputed percentile column
RAW_TO_PCTILE = {
    "total_population":           "total_population_pctile",
    "age_65_plus_pct":            "age_65_plus_pct_pctile",
    "population_growth_rate_2yr": "population_growth_rate_2yr_pctile",
    "age_45_64_pct":              "age_45_64_pct_pctile",
    "birth_rate_per_1000":        "birth_rate_per_1000_pctile",
}

print("✅ SCORE_DEFINITIONS and RAW_TO_PCTILE loaded")
print(f"   Options available: {list(SCORE_DEFINITIONS.keys())}")

✅ SCORE_DEFINITIONS and RAW_TO_PCTILE loaded
   Options available: ['attractiveness_score_opt1', 'attractiveness_score_opt2', 'attractiveness_score_opt4']


## User Question Classification

In [0]:
# ── FIX 2: Stronger intent classifier with explicit score key mapping ─────────
def classify_question_intent(user_question: str) -> dict:
    """
    Improved classifier — explicitly maps 'option 2' → 'attractiveness_score_opt2'
    and is much more aggressive about detecting explanation intent.
    """
    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a query intent classifier. Return only valid JSON, no markdown."},
            {"role": "user", "content": (
                f"Classify this question and return JSON with these exact fields:\n\n"
                f"{{\n"
                f"  \"intent\": \"surface_query\" or \"explanation\" or \"comparison\",\n"
                f"  \"score_column\": \"attractiveness_score_opt1\" or \"attractiveness_score_opt2\" or \"attractiveness_score_opt3\" or null,\n"
                f"  \"geographic_level\": \"zip\" or \"county\" or \"msa\" or null,\n"
                f"  \"target_entity\": \"<specific name or top or null>\",\n"
                f"  \"needs_ranking_first\": true or false\n"
                f"}}\n\n"
                f"CLASSIFICATION RULES:\n"
                f"- intent = 'explanation' if question contains: why, what makes, explain, reason, factor, drive, cause\n"
                f"- intent = 'comparison' if question contains: compare, vs, versus, difference, how does X differ\n"
                f"- intent = 'surface_query' for all others (what is, which, how many, average, highest, lowest)\n"
                f"- score_column: 'option 1' or 'opt1' → attractiveness_score_opt1, "
                f"'option 2' or 'opt2' → attractiveness_score_opt2, "
                f"'option 3' or 'opt3' → attractiveness_score_opt3\n"
                f"- target_entity: extract the specific MSA/county/zip name if mentioned, "
                f"or 'top' if asking about highest scoring\n\n"
                f"Question: \"{user_question}\""
            )}
        ],
        max_tokens=200,
        temperature=0.0
    )

    raw = response.choices[0].message.content.strip()
    # Strip markdown fences if LLM adds them despite instructions
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        intent = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: crude keyword detection so we never silently fail
        q_lower = user_question.lower()
        intent = {
            "intent": (
                "explanation" if any(w in q_lower for w in ["why", "what makes", "explain", "reason", "factor"])
                else "comparison" if any(w in q_lower for w in ["compare", " vs ", "versus", "differ"])
                else "surface_query"
            ),
            "score_column": (
                "attractiveness_score_opt1" if "option 1" in q_lower or "opt1" in q_lower
                else "attractiveness_score_opt2" if "option 2" in q_lower or "opt2" in q_lower
                else "attractiveness_score_opt3" if "option 3" in q_lower or "opt3" in q_lower
                else None
            ),
            "geographic_level": (
                "zip"    if re.search(r'\b\d{5}\b', user_question)
                else "county" if "county" in q_lower
                else "msa"
            ),
            "target_entity": "top",
            "needs_ranking_first": False
        }

    print(f"  Raw classifier output: {raw[:120]}")
    return intent

In [0]:
def execute_llm_generated_code(df: pd.DataFrame, user_question: str) -> str:
    """
    Pattern: LLM sees column schema only → writes Pandas code → 
    code runs locally against full dataframe → result sent back to LLM → 
    LLM writes final answer.
    """

    # ── Step 1: Give LLM the schema + sample, NOT the full data ──────────────
    schema_context = (
        f"You have access to a pandas DataFrame called `df` with the following structure:\n\n"
        f"COLUMNS:\n{df.dtypes.to_string()}\n\n"
        f"SHAPE: {df.shape[0]} rows x {df.shape[1]} columns\n\n"
        f"SAMPLE (first 3 rows):\n{df.head(3).to_csv(index=False)}\n\n"
        f"STATISTICS:\n{df.describe().to_string()}"
    )

    # ── Step 2: Ask LLM to write Pandas code to answer the question ──────────
    code_generation_prompt = (
        f"{schema_context}\n\n"
        f"User question: \"{user_question}\"\n\n"
        f"Write Python/Pandas code to answer this question using the DataFrame `df`.\n"
        f"Return ONLY executable Python code, no explanation, no markdown fences.\n"
        f"Store your final answer in a variable called `result`.\n"
        f"If the exact column doesn't exist, use the closest available columns and\n"
        f"add a comment explaining your assumption."
    )

    code_response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a Python/Pandas expert. Return only raw executable code."},
            {"role": "user",   "content": code_generation_prompt}
        ],
        max_tokens=512,
        temperature=0.1
    )

    generated_code = code_response.choices[0].message.content

    # ── Step 3: Execute the generated code against the REAL dataframe ─────────
    local_vars = {"df": df}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", "No result variable found.")
    except Exception as e:
        result = f"Code execution error: {e}"

    # ── Step 4: Send result back to LLM for a natural language answer ─────────
    final_response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a strategic healthcare market analyst for Orlando Health, writing for senior executives. Every response MUST follow this exact structure:\n1. HEADLINE (one bold sentence): the direct answer or bottom-line conclusion.\n2. SUPPORTING BULLETS: 3-5 concise bullet points with the key facts, numbers, and context.\n3. STRATEGIC IMPLICATION (one bold sentence): what this means for Orlando Health.\n4. SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next.\nUse dashes only in hyphenated words (e.g. part-time); use colons and semicolons to separate clauses, never dashes."},
            {"role": "user",   "content": (
                f"Original question: \"{user_question}\"\\n\\n"
                f"The Pandas code returned this result:\\n{result}\\n\\n"
                f"Answer using this exact structure:\\n"
                f"HEADLINE: One bold sentence with the direct answer.\\n"
                f"SUPPORTING BULLETS: 3-5 bullets with key facts and numbers.\\n"
                f"STRATEGIC IMPLICATION: One bold sentence on what this means for Orlando Health.\\n"
                f"SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next."
            )}
        ],
        max_tokens=400,
        temperature=0.3
    )

    return final_response.choices[0].message.content

## Explanation Scenarios

In [0]:
# ── NEW: Explanation engine ───────────────────────────────────────────────────
def explain_attractiveness_score(
    df: pd.DataFrame,
    user_question: str,
    intent: dict
) -> str:
    """
    Called when intent == 'explanation'.
    Steps:
      1. Find the target entity (e.g. top MSA, or named MSA)
      2. Pull its raw component values
      3. Compare each component to dataset benchmarks
      4. Ask LLM to narrate WHY the score is what it is
    """

    score_col = intent.get("score_column")
    geo_level = intent.get("geographic_level", "msa")
    target    = intent.get("target_entity", "top")

    if not score_col or score_col not in SCORE_DEFINITIONS:
        # Fallback: can't explain without knowing which score
        return execute_llm_generated_code(df, user_question)

    score_def    = SCORE_DEFINITIONS[score_col]
    component_cols = list(score_def["components"].keys())

    # ── Step 1: Find the target row(s) ───────────────────────────────────────
    geo_col = {
        "msa":    "msa_name",
        "county": "county_name",
        "zip":    "zipcode"
    }.get(geo_level, "msa_name")

    # Aggregate to geo level (mean for rates/pcts, sum for totals)
    sum_cols  = [c for c in component_cols if c in df.columns and
                 any(k in c for k in ["population", "income", "unemployed", "migration"])]
    mean_cols = [c for c in component_cols if c in df.columns and c not in sum_cols]

    agg_dict = {score_col: "mean"}
    for c in component_cols:
        if c in df.columns:
            agg_dict[c] = "sum" if c in sum_cols else "mean"

    # Only aggregate if geo_col exists and we need rollup
    if geo_col in df.columns and geo_level != "zip":
        grouped = df.groupby(geo_col).agg(agg_dict).reset_index()
    else:
        grouped = df.copy()

    # Resolve target entity
    if target == "top":
        target_row = grouped.loc[grouped[score_col].idxmax()]
        target_name = target_row[geo_col]
    else:
        mask = grouped[geo_col].str.contains(target, case=False, na=False)
        if mask.sum() == 0:
            return execute_llm_generated_code(df, user_question)
        target_row  = grouped[mask].iloc[0]
        target_name = target_row[geo_col]

    # ── Step 2: Build component comparison vs dataset benchmarks ─────────────
    benchmark_stats = grouped[component_cols].agg(["mean", "min", "max", "median"])

    component_analysis = []
    for col, meta in score_def["components"].items():
        if col not in grouped.columns:
            continue

        val      = target_row[col]
        col_mean = benchmark_stats.loc["mean", col]
        col_min  = benchmark_stats.loc["min",  col]
        col_max  = benchmark_stats.loc["max",  col]

        # Determine relative strength
        pct_rank = (grouped[col] <= val).mean() * 100  # percentile rank

        if meta["direction"] == "higher_is_better":
            strength = "STRONG ↑" if pct_rank >= 75 else "MODERATE" if pct_rank >= 40 else "WEAK ↓"
        else:  # lower_is_better
            strength = "STRONG ↑" if pct_rank <= 25 else "MODERATE" if pct_rank <= 60 else "WEAK ↓"

        component_analysis.append({
            "factor":      meta["label"],
            "column":      col,
            "weight":      f"{meta['weight']*100:.0f}%",
            "value":       round(float(val), 2),           # ← cast to float
            "dataset_avg": round(float(col_mean), 2),      # ← cast to float
            "percentile":  f"{float(pct_rank):.0f}th",     # ← cast to float
            "strength":    strength,
            "direction":   meta["direction"]
        })

    # ── Step 3: Ask LLM to write the narrative explanation ───────────────────
    explanation_prompt = (
        f"You are a strategic market analyst for Orlando Health.\n\n"
        f"The user asked: \"{user_question}\"\n\n"
        f"The answer is: **{target_name}** has the "
        f"{'highest' if target == 'top' else ''} "
        f"{score_def['description']} ({score_col}) = {round(float(target_row[score_col]), 3)}.\n\n"
        f"Here is a factor-by-factor breakdown comparing {target_name} against all "
        f"{geo_level.upper()}-level markets in the dataset:\n\n"
        f"{json.dumps(component_analysis, indent=2, cls=NumpyEncoder)}\n\n"   # ← NumpyEncoder here
        f"Score formula context:\n"
        f"{score_def['description']}\n"
        f"Component weights: "
        f"{json.dumps({m['label']: m['weight'] for m in score_def['components'].values()}, cls=NumpyEncoder)}\n\n"  # ← and here
        f"Write a clear, executive-level explanation (3-5 sentences) that:\n"
        f"1. States the CONCLUSION first (which MSA/market won and its score)\n"
        f"2. Explains the TOP 2-3 STRONGEST factors driving the high score\n"
        f"   - use the percentile rank and comparison to dataset average\n"
        f"3. Notes any WEAK factors that partially held the score back (if any)\n"
        f"4. Ends with one sentence on the strategic implication for Orlando Health\n\n"
        f"Use plain business language. Be specific with numbers. "
        f"Use dashes only in hyphenated words; use colons and semicolons to separate clauses.\n"
        f"End with a short section titled 'Suggested Follow-Up Questions:' "
        f"listing 2-3 specific questions the executive could ask next."
    )

    explanation_response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a strategic healthcare market analyst for Orlando Health, writing for senior executives. Every response MUST follow this exact structure:\n1. HEADLINE (one bold sentence): the direct answer or bottom-line conclusion.\n2. SUPPORTING BULLETS: 3-5 concise bullet points with the key facts, numbers, and context.\n3. STRATEGIC IMPLICATION (one bold sentence): what this means for Orlando Health.\n4. SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next.\nUse dashes only in hyphenated words (e.g. part-time); use colons and semicolons to separate clauses, never dashes."},
            {"role": "user",   "content": explanation_prompt}
        ],
        max_tokens=400,
        temperature=0.3
    )

    return explanation_response.choices[0].message.content

## Comparison Scenarios

In [0]:
def handle_comparison(df: pd.DataFrame, user_question: str, intent: dict) -> str:
    """
    Comparison handler that pre-fetches both entity rows so the LLM
    never has to guess MSA name strings against the dataframe.
    """
    score_col = intent.get("score_column")
    geo_level = intent.get("geographic_level", "msa")
    geo_col   = {"msa": "msa_name", "county": "county_name", "zip": "zipcode"}.get(geo_level, "msa_name")

    # ── Step 1: Aggregate to geo level ───────────────────────────────────────
    if geo_col in df.columns and geo_level != "zip":
        grouped = df.groupby(geo_col)[score_col].mean().reset_index()
        grouped.columns = [geo_col, score_col]
    else:
        grouped = df[[geo_col, score_col]].copy()

    # ── Step 2: Ask LLM to identify the two entity names from the question ───
    #    Give it the ACTUAL list of names from the dataframe so it can match exactly
    available_names = grouped[geo_col].dropna().unique().tolist()

    name_resolution = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a string matcher. Return only valid JSON."},
            {"role": "user", "content": (
                f"From this list of available names:\n{available_names}\n\n"
                f"Find the TWO names that best match what the user is asking about in this question:\n"
                f"\"{user_question}\"\n\n"
                f"Return JSON: {{\"entity_1\": \"<exact name from list>\", \"entity_2\": \"<exact name from list>\"}}"
            )}
        ],
        max_tokens=100,
        temperature=0.0
    )

    try:
        raw = name_resolution.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        entities = json.loads(raw)
        name_1 = entities["entity_1"]
        name_2 = entities["entity_2"]
    except Exception:
        # Fallback: just explain the top 2
        top2 = grouped.nlargest(2, score_col)[geo_col].tolist()
        name_1, name_2 = top2[0], top2[1]

    print(f"  Resolved entities: '{name_1}' vs '{name_2}'")

    # ── Step 3: Pull pre-aggregated rows for both entities ────────────────────
    row_1 = grouped[grouped[geo_col] == name_1].iloc[0] if name_1 in grouped[geo_col].values else None
    row_2 = grouped[grouped[geo_col] == name_2].iloc[0] if name_2 in grouped[geo_col].values else None

    if row_1 is None or row_2 is None:
        return execute_llm_generated_code(df, user_question)

    score_1 = round(float(row_1[score_col]), 3)
    score_2 = round(float(row_2[score_col]), 3)
    winner  = name_1 if score_1 >= score_2 else name_2
    diff    = round(abs(score_1 - score_2), 3)

    # ── Step 4: Pull component breakdowns for BOTH entities ───────────────────
    intent_1 = {**intent, "target_entity": name_1}
    intent_2 = {**intent, "target_entity": name_2}

    explanation_1 = explain_attractiveness_score(df, f"Why does {name_1} score the way it does?", intent_1)
    explanation_2 = explain_attractiveness_score(df, f"Why does {name_2} score the way it does?", intent_2)

    # ── Step 5: Synthesize into a side-by-side comparison ────────────────────
    synthesis = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a strategic healthcare market analyst for Orlando Health, writing for senior executives. Every response MUST follow this exact structure:\n1. HEADLINE (one bold sentence): the direct answer or bottom-line conclusion.\n2. SUPPORTING BULLETS: 3-5 concise bullet points with the key facts, numbers, and context.\n3. STRATEGIC IMPLICATION (one bold sentence): what this means for Orlando Health.\n4. SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next.\nUse dashes only in hyphenated words (e.g. part-time); use colons and semicolons to separate clauses, never dashes."},
            {"role": "user", "content": (
                f"Compare these two markets on {score_col}:\n\n"
                f"{name_1} score: {score_1}\n"
                f"{name_2} score: {score_2}\n"
                f"Score difference: {diff} (winner: {winner})\n\n"
                f"--- {name_1} factor analysis ---\n{explanation_1}\n\n"
                f"--- {name_2} factor analysis ---\n{explanation_2}\n\n"
                f"Write a 4-6 sentence executive comparison that:\n"
                f"1. States which market wins and by how much\n"
                f"2. Identifies the 1-2 factors where {name_1} leads\n"
                f"3. Identifies the 1-2 factors where {name_2} leads\n"
                f"4. Gives a final strategic recommendation for Orlando Health\n"
                f"Be specific with numbers. "
                f"Use dashes only in hyphenated words; use colons and semicolons to separate clauses.\n"
                f"End with a short section titled 'Suggested Follow-Up Questions:' "
                f"listing 2-3 specific questions the executive could ask next."
            )}
        ],
        max_tokens=450,
        temperature=0.3
    )

    return synthesis.choices[0].message.content

## What IF Scanrios

In [0]:
# ── NEW FUNCTION 1: Detect if question contains a "what if" scenario ─────────
def detect_whatif_scenario(user_question: str, df: pd.DataFrame) -> dict:
    """
    Extract the hypothetical change from the question.
    Returns a dict describing what to change, for which entity.
    """
    available_cols    = df.dtypes.to_string()
    available_msas    = df["msa_name"].dropna().unique().tolist() if "msa_name" in df.columns else []

    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a scenario parser. Return only valid JSON."},
            {"role": "user", "content": (
                f"Extract the hypothetical scenario from this question and return JSON:\n\n"
                f"{{\n"
                f"  \"is_whatif\": true or false,\n"
                f"  \"changes\": [\n"
                f"    {{\n"
                f"      \"entity\":     \"<exact MSA/county/zip name being changed>\",\n"
                f"      \"column\":     \"<exact dataframe column name to modify>\",\n"
                f"      \"operation\":  \"multiply\" or \"add\" or \"set\",\n"
                f"      \"value\":      <numeric value>\n"
                f"    }}\n"
                f"  ],\n"
                f"  \"score_column\": \"attractiveness_score_opt1\" or "
                f"\"attractiveness_score_opt2\" or \"attractiveness_score_opt3\",\n"
                f"  \"geographic_level\": \"msa\" or \"county\" or \"zip\"\n"
                f"}}\n\n"
                f"Available DataFrame columns:\n{available_cols}\n\n"
                f"Available MSA names: {available_msas[:20]}\n\n"
                f"Examples of operation mapping:\n"
                f"  'will double'         → multiply 2\n"
                f"  'will triple'         → multiply 3\n"
                f"  'will grow by 50%'    → multiply 1.5\n"
                f"  'will increase by X'  → add X\n"
                f"  'will remain the same'→ (no change needed, skip)\n"
                f"  'will drop to X'      → set X\n\n"
                f"Question: \"{user_question}\""
            )}
        ],
        max_tokens=300,
        temperature=0.0
    )

    try:
        raw = response.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception:
        return {"is_whatif": False, "changes": []}

In [0]:
def apply_whatif_scenario(df: pd.DataFrame, scenario: dict) -> pd.DataFrame:
    """
    Accepts EITHER raw column names ('total_population') OR
    percentile column names ('total_population_pctile') in the scenario.
    Always modifies the RAW column — rescore_after_scenario() will
    recompute the percentile from scratch.
    """
    df_scenario = df.copy()
    geo_col = {"msa": "msa_name", "county": "county_name", "zip": "zipcode"}.get(
                scenario.get("geographic_level", "msa"), "msa_name")

    for change in scenario.get("changes", []):
        entity    = change["entity"]
        col       = change["column"]
        operation = change["operation"]
        value     = change["value"]

        # ── Auto-resolve: if LLM gave percentile col, use the raw col instead ─
        if col.endswith("_pctile"):
            raw_col = next(
                (raw for raw, pct in RAW_TO_PCTILE.items() if pct == col),
                col
            )
            print(f"  ℹ️  Resolved '{col}' → raw column '{raw_col}'")
            col = raw_col

        if col not in df_scenario.columns:
            print(f"  ⚠️  Column '{col}' not found — skipping")
            continue

        mask = df_scenario[geo_col].str.contains(entity, case=False, na=False)
        if mask.sum() == 0:
            print(f"  ⚠️  Entity '{entity}' not found — skipping")
            continue

        before_sample = round(float(df_scenario.loc[mask, col].iloc[0]), 2)
        before_total  = round(float(df_scenario.loc[mask, col].sum()), 0)

        if operation == "multiply":
            df_scenario.loc[mask, col] = df_scenario.loc[mask, col] * value
        elif operation == "add":
            df_scenario.loc[mask, col] = df_scenario.loc[mask, col] + value
        elif operation == "set":
            df_scenario.loc[mask, col] = value

        after_sample = round(float(df_scenario.loc[mask, col].iloc[0]), 2)
        after_total  = round(float(df_scenario.loc[mask, col].sum()), 0)

        print(f"  ✅ Applied: {entity} | {col} {operation} {value} "
              f"({mask.sum()} zip rows)")
        print(f"     Sample zip:  {before_sample:,.1f} → {after_sample:,.1f}")
        print(f"     MSA total:   {before_total:,.0f} → {after_total:,.0f}")

    return df_scenario

In [0]:
def rescore_after_scenario(
    df_original: pd.DataFrame,
    df_scenario: pd.DataFrame,
    score_col: str,
    geo_col: str
) -> pd.DataFrame:
    """
    Since percentile columns already exist (0-99 scale), rescoring is:
    1. Recompute the percentile of the changed raw value against
       the ORIGINAL distribution
    2. Apply weights directly — same formula as your Option 2 notebook:
       score = Σ(weight_i × pctile_i) / 99 × 100
    No min-max normalization needed. No clipping issues.
    """
    if score_col not in SCORE_DEFINITIONS:
        return df_scenario

    score_def   = SCORE_DEFINITIONS[score_col]
    df_rescored = df_scenario.copy()

    for pctile_col, meta in score_def["components"].items():
        # Find the corresponding raw column for this percentile col
        raw_col = next(
            (raw for raw, pct in RAW_TO_PCTILE.items() if pct == pctile_col),
            None
        )

        if raw_col is None or raw_col not in df_rescored.columns:
            continue

        # Recompute percentile of the (possibly changed) raw value
        # against the ORIGINAL distribution — same as national percentile logic
        original_raw_vals = df_original[raw_col].dropna().values

        def compute_pctile(v, orig_vals=original_raw_vals,
                           direction=meta["direction"]):
            if direction == "higher_is_better":
                pct = (orig_vals <= v).mean() * 99   # 0-99 scale
            else:
                pct = (orig_vals >= v).mean() * 99
            return round(float(np.clip(pct, 0, 99)), 2)

        df_rescored[pctile_col] = df_rescored[raw_col].apply(compute_pctile)

    # Recompute score — exact same formula as your Option 2 notebook:
    # score = Σ(weight_i × pctile_i) / 99 × 100
    weighted_sum = sum(
        df_rescored[pctile_col] * meta["weight"]
        for pctile_col, meta in score_def["components"].items()
        if pctile_col in df_rescored.columns
    )
    df_rescored[score_col] = (weighted_sum / 99 * 100).round(2)

    return df_rescored

In [0]:
# ── UPDATE handle_whatif() — pass df_original to rescore ─────────────────────
def handle_whatif(df: pd.DataFrame, user_question: str, scenario: dict) -> str:

    score_col = scenario.get("score_column", "attractiveness_score_opt2")
    geo_level = scenario.get("geographic_level", "msa")
    geo_col   = {"msa": "msa_name", "county": "county_name", "zip": "zipcode"}.get(
                    geo_level, "msa_name")

    entities_changed = list({c["entity"] for c in scenario.get("changes", [])})

    # ── Before ────────────────────────────────────────────────────────────────
    grouped_before = (df.groupby(geo_col)[score_col].mean().reset_index()
                      if geo_level != "zip" else df[[geo_col, score_col]].copy())

    # ── Apply scenario & re-score with FIXED anchors from original df ─────────
    df_scenario  = apply_whatif_scenario(df, scenario)
    df_scenario  = rescore_after_scenario(df, df_scenario, score_col, geo_col)
    #                                     ^^
    #                                     Pass original df as anchor

    # ── After ─────────────────────────────────────────────────────────────────
    grouped_after = (df_scenario.groupby(geo_col)[score_col].mean().reset_index()
                     if geo_level != "zip" else df_scenario[[geo_col, score_col]].copy())

    # ── Build comparison table ────────────────────────────────────────────────
    comparison_rows = []
    for entity in entities_changed:
        mask   = grouped_before[geo_col].str.contains(entity, case=False, na=False)
        mask_a = grouped_after[geo_col].str.contains(entity, case=False, na=False)

        if mask.sum() == 0 or mask_a.sum() == 0:
            continue

        score_before = round(float(grouped_before[mask][score_col].iloc[0]), 4)
        score_after  = round(float(grouped_after[mask_a][score_col].iloc[0]), 4)
        exact_name   = grouped_before[mask][geo_col].iloc[0]

        comparison_rows.append({
            "entity":       exact_name,
            "score_before": score_before,
            "score_after":  score_after,
            "delta":        round(score_after - score_before, 4),
            "delta_pct":    round((score_after - score_before) / score_before * 100, 1)
        })

    # Rankings before vs after
    rank_before = grouped_before.sort_values(score_col, ascending=False).reset_index(drop=True)
    rank_after  = grouped_after.sort_values(score_col, ascending=False).reset_index(drop=True)
    rank_before["rank"] = rank_before.index + 1
    rank_after["rank"]  = rank_after.index + 1

    rank_changes = []
    for row in comparison_rows:
        rb = rank_before[rank_before[geo_col].str.contains(row["entity"], case=False, na=False)]
        ra = rank_after[rank_after[geo_col].str.contains(row["entity"], case=False, na=False)]
        if len(rb) > 0 and len(ra) > 0:
            rank_changes.append({
                "entity":      row["entity"],
                "rank_before": int(rb["rank"].iloc[0]),
                "rank_after":  int(ra["rank"].iloc[0]),
                "rank_change": int(rb["rank"].iloc[0]) - int(ra["rank"].iloc[0])
                # positive = moved up, negative = moved down
            })

    # ── Final narrative ───────────────────────────────────────────────────────
    synthesis = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": "You are a strategic healthcare market analyst for Orlando Health, writing for senior executives. Every response MUST follow this exact structure:\n1. HEADLINE (one bold sentence): the direct answer or bottom-line conclusion.\n2. SUPPORTING BULLETS: 3-5 concise bullet points with the key facts, numbers, and context.\n3. STRATEGIC IMPLICATION (one bold sentence): what this means for Orlando Health.\n4. SUGGESTED FOLLOW-UP QUESTIONS: 2-3 specific questions the executive could ask next.\nUse dashes only in hyphenated words (e.g. part-time); use colons and semicolons to separate clauses, never dashes."},
            {"role": "user", "content": (
                f"The user asked: \"{user_question}\"\n\n"
                f"IMPORTANT CONTEXT: Scores are calculated on a fixed scale anchored to the "
                f"original dataset min/max, so before and after scores ARE directly comparable. "
                f"A score increase means the market genuinely improved relative to the same baseline.\n\n"
                f"SCORE CHANGES:\n{json.dumps(comparison_rows, indent=2, cls=NumpyEncoder)}\n\n"
                f"RANKING CHANGES:\n{json.dumps(rank_changes, indent=2, cls=NumpyEncoder)}\n\n"
                f"Top 5 markets BEFORE scenario:\n"
                f"{rank_before[[geo_col, score_col]].head(5).to_string(index=False)}\n\n"
                f"Top 5 markets AFTER scenario:\n"
                f"{rank_after[[geo_col, score_col]].head(5).to_string(index=False)}\n\n"
                f"Write a 4-6 sentence executive response that:\n"
                f"1. States what the scenario assumed\n"
                f"2. Shows exact score change (before → after) and whether it went up or down\n"
                f"3. States the ranking change clearly (e.g. moved from rank X to rank Y)\n"
                f"4. For ANY entity mentioned in the question, explicitly state BOTH "
                f"   their before AND after rank — never leave a rank unstated\n"
                f"5. If comparing two entities, state which one leads and by exactly "
                f"   how many rank positions and score points\n"
                f"Be specific with numbers. "
                f"Use dashes only in hyphenated words; use colons and semicolons to separate clauses.\n"
                f"End with a short section titled 'Suggested Follow-Up Questions:' "
                f"listing 2-3 specific questions the executive could ask next."
            )}
        ],
        max_tokens=450,
        temperature=0.3
    )

    return synthesis.choices[0].message.content

## Overall Function

In [0]:
# ── UPDATE ask_agent() to include What-If routing ────────────────────────────
def ask_agent(df: pd.DataFrame, user_question: str) -> str:

    print(f"\n{'='*65}")
    print(f"QUESTION: {user_question}")
    print('='*65)

    # ── Check for What-If FIRST before other intent detection ─────────────────
    whatif_keywords = ["what if", "what happens if", "suppose", "assume", "hypothetically",
                       "if we know", "if the", "would change if", "scenario"]

    if any(kw in user_question.lower() for kw in whatif_keywords):
        scenario = detect_whatif_scenario(user_question, df)
        print(f"  What-If scenario detected: {scenario.get('changes')}")

        if scenario.get("is_whatif"):
            answer = handle_whatif(df, user_question, scenario)
            print(f"\nANSWER:\n{answer}")
            return answer

    # ── Otherwise normal routing ──────────────────────────────────────────────
    intent = classify_question_intent(user_question)
    print(f"Intent detected: {intent.get('intent')} | "
          f"Score: {intent.get('score_column')} | "
          f"Level: {intent.get('geographic_level')} | "
          f"Target: {intent.get('target_entity')}")

    if intent.get("intent") == "explanation":
        answer = explain_attractiveness_score(df, user_question, intent)
    elif intent.get("intent") == "comparison":
        answer = handle_comparison(df, user_question, intent)
    else:
        answer = execute_llm_generated_code(df, user_question)

    print(f"\nANSWER:\n{answer}")
    return answer

In [0]:
import numpy as np

# ── Add this custom encoder right after your imports ─────────────────────────
class NumpyEncoder(json.JSONEncoder):
    """Converts numpy int64/float64 → native Python types for json.dumps."""
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, (np.ndarray,)):
            return obj.tolist()
        return super().default(obj)

## User Demo

In [0]:
# ── Test all question types ───────────────────────────────────────────────────
questions = [
    # Surface queries (routes to original code — no change)
    #"Which MSA has the highest average attractiveness score under option 2?",
    #"What is the average attractiveness score under option 2 for Birmingham-Hoover MSA?",

    # Explanation queries (routes to new explain engine)
    #"Why does Huntsville, AL MSA have the highest attractiveness score under option 2?",
    #"What makes the top MSA under option 2 score so high?",
    "Explain which zip code has the highest ranking of option 2 and why is that",
    "Explain which zip code in Alabama has the highest ranking of option 2 and why is that",


    # Comparison (uses both engines)
    "Compare Birmingham-Hoover MSA vs Huntsville MSA on attractiveness score option 2 and explain the difference.",

    # What if scenarios
    "What if Birmingham-Hoover MSA population growth rate catches up to Huntsville's over the next 3 years?"
]

for q in questions:
    ask_agent(df, q)


QUESTION: Explain which zip code has the highest ranking of option 2 and why is that
  Raw classifier output: {
  "intent": "explanation",
  "score_column": "attractiveness_score_opt2",
  "geographic_level": "zip",
  "target_entit
Intent detected: explanation | Score: attractiveness_score_opt2 | Level: zip | Target: top

ANSWER:
**ZIP code 32162 ranks highest under Option 2, Neil's expert-weighted 5-factor model, with an attractiveness score of 88.1 out of 100.**

- **Total Population (50% weight):** 32162 scores at the 97th percentile, with a raw percentile value of 95.0 against a dataset average of 54.53; this single factor, carrying the heaviest weight, is the primary engine behind the top ranking.
- **Age 45 to 64 Share (10% weight):** The ZIP scores at the 91st percentile with a value of 90.0, nearly double the dataset average of 48.98; this signals a large pre-senior cohort approaching peak healthcare utilization years.
- **Age 65+ Share (20% weight):** At the 77th percentile wi

[Trace(trace_id=tr-fe91639f60eae035c84cc62038211522), Trace(trace_id=tr-6bee4a81a5829319a2c6039b53463250), Trace(trace_id=tr-bcbdd0b2671ffc093a316e0e30b2cdeb), Trace(trace_id=tr-3127053583eeb4921341b0f3f125dbc6), Trace(trace_id=tr-c796e53c7e2709f2c8777d9425520dcd), Trace(trace_id=tr-950af5a0f4e8ee46cc2aaf4ba1fbf70a), Trace(trace_id=tr-b71bbb00c3817c9714370187e6f7f5b2), Trace(trace_id=tr-cb906eb9237f901db399164f60246ae4), Trace(trace_id=tr-da712fd1d45a1c4801db1a0d863c30c0), Trace(trace_id=tr-312af162988eed4632ce02ab3f35c24c)]

In [0]:
whatif_questions = [
    "What if Huntsville MSA total population doubles in the next 2 years — how would its option 2 score change?",
    "What if Birmingham-Hoover MSA population growth rate drops to zero — would it still rank above Huntsville on option 2?",
    "What if we know that in the next 2 years Huntsville MSA total population will double while Birmingham-Hoover MSA total population remains the same — who leads on option 2?",
]

for q in whatif_questions:
    ask_agent(df, q)


QUESTION: What if Huntsville MSA total population doubles in the next 2 years — how would its option 2 score change?
  What-If scenario detected: [{'entity': 'Huntsville', 'column': 'total_population', 'operation': 'multiply', 'value': 2}]
  ✅ Applied: Huntsville | total_population multiply 2 (32 zip rows)
     Sample zip:  29,969.0 → 59,938.0
     MSA total:   516,791 → 1,033,582

ANSWER:
**A population doubling in Huntsville MSA would meaningfully improve its Option 2 attractiveness score, lifting it from 56.13 to 59.98 and vaulting it eight positions up the market rankings.**

- The scenario assumed Huntsville MSA's total population doubles within two years, testing the sensitivity of its attractiveness score to a dramatic demographic expansion.
- Huntsville's score rises from 56.13 to 59.98, a gain of 3.84 points (6.8%), reflecting genuine improvement against the same fixed baseline used across all markets.
- Its rank improves from 20th to 12th, an eight-position climb, signaling 

[Trace(trace_id=tr-4fdda5f3007bd73818cf548df8156c12), Trace(trace_id=tr-a511bfb449fd09f0283ed246daaaf01a), Trace(trace_id=tr-f3f62b1f669a8d8a0ba34b48d5b9d51c), Trace(trace_id=tr-792d3716ed0c0ea9c2ef419844c35a88), Trace(trace_id=tr-6d3fc89f524b6200bd3ef3be646821d7), Trace(trace_id=tr-35f5a8004091e5984347992d357f1240)]